### Pass@k

In [ ]:
import pandas as pd

df = pd.read_csv("/path/to/DataSciBench/evaluation_results/glm-4-flash_results.csv")

In [15]:
df.head(10)

,model_name,run_id,data_name,task_name,metric_name,function_name,result_value,result_cr,result_type
0,glm-4-flash,9,human_8,Data cleaning and preprocessing,Data Cleaning Quality,Data Cleaning Completeness,False,1.0,single task (bool)
1,glm-4-flash,9,human_8,Data mining,Clustering Silhouette Score,Silhouette Score,Error,0.0,single task
2,glm-4-flash,9,human_8,Pattern recognition,Association Rule Strength (Lift),Pattern Recognition Validity,Error,0.0,single task
3,glm-4-flash,9,human_8,Data visualization,Feature Importance Scores,Plot Validity,Error,0.0,single task
4,glm-4-flash,9,human_8,Interpretability and report generation,Report Completeness,Report Completeness,False,1.0,single task (bool)
5,glm-4-flash,9,human_8,Completion Rate,CR,CR,0.2,0.2,Completion Rate
6,glm-4-flash,8,human_8,Data cleaning and preprocessing,Data Cleaning Quality,Data Cleaning Completeness,False,1.0,single task (bool)
7,glm-4-flash,8,human_8,Data mining,Clustering Silhouette Score,Silhouette Score,Error,0.0,single task
8,glm-4-flash,8,human_8,Pattern recognition,Association Rule Strength (Lift),Pattern Recognition Validity,Error,0.0,single task
9,glm-4-flash,8,human_8,Data visualization,Feature Importance Scores,Plot Validity,Error,0.0,single task


In [16]:
# get all the row where result_type is Completion Rate

df_crs = df[df['result_type'] == 'Completion Rate']

In [17]:
success_dict = {}
for index, row in df_crs.iterrows():
    if row['data_name'] not in success_dict:
        success_dict[row['data_name']] = 0
    if row['result_cr'] == 1:
        success_dict[row['data_name']] += 1


In [18]:
# count the number of success and total
success_counter = 0
fail_counter = 0
total_counter = 0
for key in success_dict:
    total_counter += 1
    if success_dict[key] == "Success":
        success_counter += 1
    if success_dict[key] == "Fail":
        fail_counter += 1

print("Pass@1: ", success_counter/total_counter)
print("Pass@1 (Fail): ", fail_counter/total_counter)

Pass@1:  0.1875
Pass@1:  0.8125


In [19]:
# count the number of success and total for data_name starting with "human"
success_counter = 0
total_counter = 0
for key in success_dict:
    if key.startswith("human"):
        total_counter += 1
        if success_dict[key] == "Success":
            success_counter += 1

print("Pass@1 for data_name starting with 'human': ", success_counter/total_counter)

Pass@1 for data_name starting with 'human':  0.30434782608695654


In [20]:
# count the number of success and total for data_name starting with "csv"
success_counter = 0
total_counter = 0
for key in success_dict:
    if key.startswith("csv"):
        total_counter += 1
        if success_dict[key] == "Success":
            success_counter += 1

print("Pass@1 for data_name starting with 'csv': ", success_counter/total_counter)

Pass@1 for data_name starting with 'csv':  0.11764705882352941


In [21]:
# count the number of success and total for data_name starting with "dl"
success_counter = 0
total_counter = 0
for key in success_dict:
    if key.startswith("dl"):
        total_counter += 1
        if success_dict[key] == "Success":
            success_counter += 1

print("Pass@1 for data_name starting with 'dl': ", success_counter/total_counter)

Pass@1 for data_name starting with 'dl':  0.0


In [22]:
# calculate average completion rate for df_crs
total_cr = 0
for index, row in df_crs.iterrows():
    total_cr += row['result_cr']

print("Average completion rate: ", total_cr/len(df_crs))

Average completion rate:  0.42362058080808074


## Calcualte CR of Top-k functions

In [29]:
top_5_functions_dict = [
    {"function_name": "Data Quality Score", "task_name": "Data cleaning and preprocessing"},
    {"function_name": "Plot Validity", "task_name": "Data visualization"},
    {"function_name": "Data Accuracy", "task_name": "Data exploration and statistics understand"},
    {"function_name": "Visualization Completeness", "task_name": "Data visualization"},
    {"function_name": "Model Accuracy", "task_name": "Predictive modeling"}
]

In [30]:
# calculate average completion rate for each (funciotn, task) in top_5_functions_dict
for function_dict in top_5_functions_dict:
    # print("function_name: ", function_dict['function_name'], "task_name: ", function_dict['task_name'])
    total_cr = 0
    count = 0
    for index, row in df.iterrows():
        # print(row['function_name'])
        try:
            if row['function_name'] == function_dict['function_name'] and row['task_name'] == function_dict['task_name']:
                total_cr += row['result_cr']
                count += 2
        except:
            print(row)
    print("Average completion rate for (", function_dict['function_name'], ", ", function_dict['task_name'], "): ", total_cr/count)

Average completion rate for ( Data Quality Score ,  Data cleaning and preprocessing ):  0.6979166666666666
Average completion rate for ( Plot Validity ,  Data visualization ):  0.0
Average completion rate for ( Data Accuracy ,  Data exploration and statistics understand ):  0.4074074074074074
Average completion rate for ( Visualization Completeness ,  Data visualization ):  0.35135135135135137
Average completion rate for ( Model Accuracy ,  Predictive modeling ):  0.26785714285714285


## Calculate bcb Metrics

In [ ]:
# iterate over ../data and the sub-folders that startswith "bcb"
import os
import re
import pandas as pd
import numpy as np

data_dir = "/path/to/DataSciBench/data"
bcb_dir = []
for root, dirs, files in os.walk(data_dir):
    for dir in dirs:
        if dir.startswith("bcb"):
            bcb_dir.append(os.path.join(root, dir))

# read all the files that end with tmc_results.csv in all the sub-sub-folders of these sub-folders
# df_bcb = pd.DataFrame()
path_list = []
model_name_list = []
model_dict = {}
for dir in bcb_dir:
    for root, dirs, files in os.walk(dir):
        # print(root)
        data_name = root.split("/")[-1]
        for file in files:
            if file.endswith("tmc_results.jsonl"):
                path_to_file = os.path.join(root, file)
                model_name = file.split("_tmc_results.jsonl")[0]
                # print(model_name)
                if model_name not in model_dict:
                    model_dict[model_name] = {}

                result_list = []
                null = 0
                true = 1
                false = 0
                NaN = 0
                with open(path_to_file, 'r') as f:
                    for line in f:
                        result_list.append(eval(line))
                result_list = result_list[:10]
                # first decide the total number of tasks
                for task in result_list[0]['TMC_results']:
                    key = task['function']
                    if key not in model_dict[model_name]:
                        model_dict[model_name][key] = {
                            "total_num": 10,
                            "success_num": 0
                        }
                    else:
                        model_dict[model_name][key]["total_num"] += 10

                for result in result_list:
                    for task in result['TMC_results']:
                        key = task['function']
                        model_dict[model_name][key]["success_num"] += task['result']

                    if "passes" not in model_dict[model_name]:
                        model_dict[model_name]["passes"] = {data_name: 0}
                    if result['cr'] == 1:
                        model_dict[model_name]["passes"][data_name] = 1

                    


In [2]:
TOTAL_DATA_NUM = 167
TOTAL_FUNCTION_NUM = 352
F1_DATA_NUM = 37
F2_DATA_NUM = 54
F3_DATA_NUM = 38
F4_DATA_NUM = 39
F5_DATA_NUM = 19 

In [ ]:
import csv
df = pd.DataFrame({
    "Model": ["Model"],
    "Pass@1": ["Pass@1"],
    # "Pass@1 (Fail)": ["Pass@1 (Fail)"],
    "Average CR": ["Average CR"],
    "VLM": ["VLM"],
    "F1": ["F1"],
    "F2": ["F2"],
    "F3": ["F3"],
    "F4": ["F4"],
    "F5": ["F5"],
    "Pass@1 (Human)": ["Pass@1 (Human)"],
    "Pass@1 (DL)": ["Pass@1 (DL)"],
    "Pass@1 (CSV)": ["Pass@1 (CSV)"],
    "Avg. CR (Human)": ["Avg. CR (Human)"],
    "Avg. CR (DL)": ["Avg. CR (DL)"],
    "Avg. CR (CSV)": ["Avg. CR (CSV)"],
})
csv_output_file = "/path/to/DataSciBench/evaluation_results/bcb_results.csv"
with open(csv_output_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(df.columns)

In [19]:
model_dict['Meta-Llama-3.1-70B-Instruct']

{'Visualization Quality': {'total_num': 20,
  'success_num': 1.0,
  'function_cr': 0.05},
 'passes': {'bcb659': 1,
  'bcb983': 1,
  'bcb1011': 1,
  'bcb162': 1,
  'bcb880': 1,
  'bcb151': 1,
  'bcb9': 1,
  'bcb393': 1,
  'bcb99': 1,
  'bcb1043': 1},
 'Plot Validity': {'total_num': 50, 'success_num': 2.0},
 'Data Integrity': {'total_num': 30, 'success_num': 3.0},
 'Prediction Accuracy': {'total_num': 10, 'success_num': 2.0},
 'Data Accuracy': {'total_num': 50, 'success_num': 8.0},
 'Visualization Completeness': {'total_num': 40, 'success_num': 9.0},
 'Report Quality': {'total_num': 10, 'success_num': 0},
 'Model Accuracy': {'total_num': 10, 'success_num': 1.0},
 'Normalization Range Check': {'total_num': 20, 'success_num': 4.0},
 'DataFrame Shape Validation': {'total_num': 20, 'success_num': 4.0},
 'Title Verification': {'total_num': 20, 'success_num': 5.0},
 'Data Uniformity': {'total_num': 10, 'success_num': 3.0}}

In [20]:
top_5_functions_dict = [
    {"function_name": "Data Quality Score", "task_name": "Data cleaning and preprocessing"},
    {"function_name": "Plot Validity", "task_name": "Data visualization"},
    {"function_name": "Data Accuracy", "task_name": "Data exploration and statistics understand"},
    {"function_name": "Visualization Completeness", "task_name": "Data visualization"},
    {"function_name": "Model Accuracy", "task_name": "Predictive modeling"}
]

In [ ]:
for model_name in model_dict:
    model = model_dict[model_name]
    total_pass = 0
    print(model['passes'])
    for key, value in model['passes'].items():
        total_pass += value
    total_pass = total_pass/TOTAL_DATA_NUM
    avg_cr = 0
    for key, value in model.items():
        if key != "passes":
            avg_cr += value["success_num"]
    avg_cr = avg_cr/(TOTAL_FUNCTION_NUM * 10)

    f1 = 0
    if 'Data Quality Score' in model:
        f1 = model['Data Quality Score']['success_num'] / (F1_DATA_NUM * 10)
    f2 = 0
    if 'Plot Validity' in model:
        f2 = model['Plot Validity']['success_num'] / (F2_DATA_NUM * 10)
    f3 = 0
    if 'Data Accuracy' in model:
        f3 = model['Data Accuracy']['success_num'] / (F3_DATA_NUM * 10)
    f4 = 0
    if 'Visualization Completeness' in model:
        f4 = model['Visualization Completeness']['success_num'] / (F4_DATA_NUM * 10)
    f5 = 0
    if 'Model Accuracy' in model:
        f5 = model['Model Accuracy']['success_num'] / (F5_DATA_NUM * 10)

    df_bcb = pd.DataFrame({
        "Model": [model_name],
        "Pass@1": [f"{total_pass:.2f}"],
        # "Pass@1 (Fail)": [f"{total_fail:.2f}"],
        "Average CR": [f"{avg_cr:.2f}"],
        "VLM": "-",
        "F1": [f"{f1:.2f}"],
        "F2": [f"{f2:.2f}"],
        "F3": [f"{f3:.2f}"],
        "F4": [f"{f4:.2f}"],
        "F5": [f"{f5:.2f}"],
    })

    df_bcb.to_csv("/path/to/DataSciBench/evaluation_results/bcb_results.csv", mode='a', header=False, index=False)

{'bcb659': 1, 'bcb983': 1, 'bcb1011': 1, 'bcb162': 1, 'bcb880': 1, 'bcb151': 1, 'bcb9': 1, 'bcb393': 1, 'bcb99': 1, 'bcb1043': 1}
{'bcb659': 1, 'bcb1011': 1, 'bcb162': 1, 'bcb9': 1, 'bcb99': 1, 'bcb1064': 1, 'bcb112': 1, 'bcb110': 1, 'bcb296': 1, 'bcb134': 1, 'bcb750': 1, 'bcb1062': 1, 'bcb44': 1, 'bcb982': 1, 'bcb908': 1, 'bcb343': 1, 'bcb1024': 1, 'bcb690': 1, 'bcb1037': 1, 'bcb815': 1, 'bcb801': 1, 'bcb864': 1, 'bcb417': 1, 'bcb432': 1, 'bcb530': 1, 'bcb47': 1, 'bcb68': 1, 'bcb319': 1, 'bcb792': 1, 'bcb38': 1, 'bcb966': 1, 'bcb225': 1, 'bcb859': 1, 'bcb337': 1}
{'bcb659': 1, 'bcb983': 1, 'bcb1011': 1, 'bcb162': 1, 'bcb527': 1, 'bcb880': 1, 'bcb430': 1, 'bcb1017': 1, 'bcb40': 1, 'bcb414': 1, 'bcb151': 1, 'bcb9': 1, 'bcb393': 1, 'bcb99': 1, 'bcb1043': 1, 'bcb866': 1, 'bcb870': 1, 'bcb468': 1, 'bcb385': 1, 'bcb345': 1, 'bcb197': 1, 'bcb61': 1, 'bcb803': 1, 'bcb251': 1, 'bcb54': 1, 'bcb209': 1, 'bcb1061': 1, 'bcb646': 1, 'bcb1064': 1, 'bcb112': 1, 'bcb871': 1, 'bcb110': 1, 'bcb660': 1, 